In [ ]:
!pip install roboflow
!pip install ultralytics #for roboflow
!pip install mlflow python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 119.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.11.0.86
    Uninstalling opencv-python-headless-4.11.0.86:
      Successfully uninstalled opencv-python-headless-4.11.0.86
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.2 MB/s eta 0:00:00
  

In [ ]:
import os
from roboflow import Roboflow
from google.colab import userdata
from ultralytics import YOLO
from ultralytics import RTDETR
import mlflow
#google drive
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
os.environ["MLFLOW_TRACKING_URI"]      = userdata.get("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("MLFLOW_TRACKING_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("MLFLOW_TRACKING_PASSWORD")

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("staves-detector")

DATA_YAML = '/content/drive/MyDrive/Stave_Project/data_for_training/wood_plank_final-2/data.yaml'
MODEL_NAME = "yolo11m.pt"
MODE = "pretrain"
RUN_NAME = f"{MODEL_NAME.replace('.pt','')}_{MODE}"

EPOCHS    = 100
IMGSZ     = 512
BATCH     = 32
LR0       = 0.01
OPTIMIZER = "AdamW"
MOSAIC    = 1.0
FLIPUD    = 0.0
FLIPLR    = 0.5

Mounted at /content/drive


In [4]:
#roboflow dataset: https://universe.roboflow.com/boarddetection-hpgqc/wood_plank_final
api_key = userdata.get('roboflow_api')
rf = Roboflow(api_key=api_key)

project = rf.workspace('boarddetection-hpgqc').project("wood_plank_final")
#ultralytics yolo format, works with ultralytics' rt-detr too
dataset = project.version(2).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to wood_plank_final-2 in yolov8:: 100%|██████████| 7248/7248 [00:01<00:00, 4793.45it/s]


In [ ]:
def on_fit_epoch_end(trainer):
    metrics = trainer.metrics
    step = trainer.epoch
    to_log = {k: float(v) for k, v in {
        "metrics/mAP50":     metrics.get("metrics/mAP50"),
        "metrics/mAP50-95":  metrics.get("metrics/mAP50-95"),
        "metrics/precision": metrics.get("metrics/precision"),
    }.items() if v is not None}
    if to_log:
        mlflow.log_metrics(to_log, step=step)


In [5]:
%cd /content/drive/MyDrive/Stave_Project/Developers/modelWeights
os.getcwd()

/content/drive/.shortcut-targets-by-id/1B5568bIq8F-Y9_NKFOOLNg1afBLZMJC2/Stave_Project/Developers/modelWeights


'/content/drive/.shortcut-targets-by-id/1B5568bIq8F-Y9_NKFOOLNg1afBLZMJC2/Stave_Project/Developers/modelWeights'

In [ ]:
run = mlflow.start_run(run_name=RUN_NAME)
try:
    mlflow.log_params({
        "model":       MODEL_NAME,
        "mode":        MODE,
        "epochs":      EPOCHS,
        "imgsz":       IMGSZ,
        "batch":       BATCH,
        "lr0":         LR0,
        "optimizer":   OPTIMIZER,
        "mosaic":      MOSAIC,
        "flipud":      FLIPUD,
        "fliplr":      FLIPLR,
        "num_classes": 1,
        "dataset":     DATA_YAML,
        "pretrained":  True,
    })

    model = YOLO(MODEL_NAME)
    model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

    results = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        lr0=LR0,
        optimizer=OPTIMIZER,
        mosaic=MOSAIC,
        flipud=FLIPUD,
        fliplr=FLIPLR,
        project="results",
        name=RUN_NAME,
    )

    save_dir = str(results.save_dir)

    for artifact_name in ["confusion_matrix.png", "PR_curve.png", "F1_curve.png", "results.png"]:
        artifact_path = os.path.join(save_dir, artifact_name)
        if os.path.exists(artifact_path):
            mlflow.log_artifact(artifact_path)

    weights_dir = os.path.join(save_dir, "weights")
    os.makedirs("weights", exist_ok=True)

    for src_name, dest_name in [("best.pt", f"best_{MODE}.pt"), ("last.pt", f"last_{MODE}.pt")]:
        src  = os.path.join(weights_dir, src_name)
        dest = os.path.join("weights", dest_name)
        if os.path.exists(src):
            import shutil
            shutil.copy2(src, dest)
            mlflow.log_artifact(dest, artifact_path="weights")

finally:
    mlflow.end_run()


In [6]:
rt_detr_save_dir = '/content/drive/MyDrive/Stave_Project/Developers/modelWeights/RTDetrPlankModel'
os.makedirs(rt_detr_save_dir, exist_ok=True)

%cd /content/drive/MyDrive/Stave_Project/Developers/modelWeights/RTDetrPlankModel
os.getcwd()

/content/drive/.shortcut-targets-by-id/1B5568bIq8F-Y9_NKFOOLNg1afBLZMJC2/Stave_Project/Developers/modelWeights/RTDetrPlankModel


'/content/drive/.shortcut-targets-by-id/1B5568bIq8F-Y9_NKFOOLNg1afBLZMJC2/Stave_Project/Developers/modelWeights/RTDetrPlankModel'

In [ ]:
RTDetr_model = RTDETR("/content/drive/MyDrive/Stave_Project/Developers/modelWeights/RTDetrPlankModel/runs/train1/weights/last.pt")
results = RTDetr_model.train(data='/content/drive/MyDrive/Stave_Project/data_for_training/wood_plank_final-2/data.yaml', epochs=130, imgsz=512, batch=10, save_dir=rt_detr_save_dir, patience=70)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: task=detect, mode=train, model=/content/drive/MyDrive/Stave_Project/Developers/modelWeights/RTDetrPlankModel/runs/train1/weights/last.pt, data=/content/drive/MyDrive/Stave_Project/data_for_training/wood_plank_final-2/data.yaml, epochs=130, time=None, patience=70, batch=10, imgsz=512, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_mas

100%|██████████| 755k/755k [00:00<00:00, 14.3MB/s]

WARNING ⚠️ no model scale passed. Assuming scale='l'.

                   from  n    params  module                                       arguments                     
  0                  -1  1     25248  ultralytics.nn.modules.block.HGStem          [3, 32, 48]                   
  1                  -1  6    155072  ultralytics.nn.modules.block.HGBlock         [48, 48, 128, 3, 6]           
  2                  -1  1      1408  ultralytics.nn.modules.conv.DWConv           [128, 128, 3, 2, 1, False]    
  3                  -1  6    839296  ultralytics.nn.modules.block.HGBlock         [128, 96, 512, 3, 6]          
  4                  -1  1      5632  ultralytics.nn.modules.conv.DWConv           [512, 512, 3, 2, 1, False]    
  5                  -1  6   1695360  ultralytics.nn.modules.block.HGBlock         [512, 192, 1024, 5, 6, True, False]
  6                  -1  6   2055808  ultralytics.nn.modules.block.HGBlock         [1024, 192, 1024, 5, 6, True, True]
  7                  -1

  9                  -1  6   6708480  ultralytics.nn.modules.block.HGBlock         [1024, 384, 2048, 5, 6, True, False]
 10                  -1  1    524800  ultralytics.nn.modules.conv.Conv             [2048, 256, 1, 1, None, 1, 1, False]
 11                  -1  1    789760  ultralytics.nn.modules.transformer.AIFI      [256, 1024, 8]                
 12                  -1  1     66048  ultralytics.nn.modules.conv.Conv             [256, 256, 1, 1]              
 13                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 14                   7  1    262656  ultralytics.nn.modules.conv.Conv             [1024, 256, 1, 1, None, 1, 1, False]
 15            [-2, -1]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 16                  -1  3   2232320  ultralytics.nn.modules.block.RepC3           [512, 256, 3]                 
 17                  -1  1     66048  ultralytics.nn.modules.conv.Conv

100%|██████████| 5.35M/5.35M [00:00<00:00, 66.9MB/s]


...

train: WARNING ⚠️ /content/drive/.shortcut-targets-by-id/1B5568bIq8F-Y9_NKFOOLNg1afBLZMJC2/Stave_Project/data_for_training/wood_plank_final-2/train/images/images-22-_jpeg_jpg.rf.080cea15f532f72bd98d58a5c07024c7.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/drive/.shortcut-targets-by-id/1B5568bIq8F-Y9_NKFOOLNg1afBLZMJC2/Stave_Project/data_for_training/wood_plank_final-2/train/images/images-22-_jpeg_jpg.rf.1c33a6146e67ab85ca76db0bc7db8da8.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/drive/.shortcut-targets-by-id/1B5568bIq8F-Y9_NKFOOLNg1afBLZMJC2/Stave_Project/data_for_training/wood_plank_final-2/train/images/images-22-_jpeg_jpg.rf.354bbc5ef1c57179025062baca8bccb9.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/drive/.shortcut-targets-by-id/1B5568bIq8F-Y9_NKFOOLNg1afBLZMJC2/Stave_Project/data_for_training/wood_plank_final-2/train/images/images-22-_jpeg_jpg.rf.70ba27e80d5fd093680aeae48af7ac95.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/d

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.8±0.3 ms, read: 0.2±0.1 MB/s, size: 61.8 KB)


val: Scanning /content/drive/.shortcut-targets-by-id/1B5568bIq8F-Y9_NKFOOLNg1afBLZMJC2/Stave_Project/data_for_training/wood_plank_final-2/valid/labels.cache... 348 images, 0 backgrounds, 0 corrupt: 100%|██████████| 348/348 [00:00<?, ?it/s]


Plotting labels to runs/detect/train/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 143 weight(decay=0.0), 206 weight(decay=0.00046875), 226 bias(decay=0.0)
Image sizes 512 train, 512 val
Using 8 dataloader workers
Logging results to runs/detect/train
Starting training for 130 epochs...

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


  0%|          | 0/325 [00:00<?, ?it/s]grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:92.)
      1/130        22G      0.453     0.3427     0.1033       1819        512:  21%|██        | 69/325 [01:00<03:55,  1.09it/s]